# 실습과제: AI 헬스케어 데이터 전처리 종합 연습

심혈관 질환 예측 데이터를 활용하여 실제 의료 데이터에서 자주 발생하는 데이터 품질 문제를 해결합니다.  
결측치, 중복 데이터, 이상치, 데이터 타입 문제를 체계적으로 처리하여 AI 모델 학습에 적합한 데이터로 변환해 보세요.

## 요구사항

1. **결측치 탐지 및 처리**
   - Age 결측치: 평균값으로 대체
   - BMI 결측치: 중앙값으로 대체 (타입 변환 후)
   - BloodPressure 결측치: 평균값으로 대체
   - Cholesterol 결측치: 중앙값으로 대체

2. **중복 데이터 처리**
   - PatientID 기준 중복 데이터 탐지
   - 최신 검사 데이터 유지 (`keep='last'`)
   - PatientID 순서로 정렬

3. **이상치 탐지 및 처리**
   - BloodPressure 이상치를 IQR 방법으로 탐지
   - IQR 범위 밖의 값을 중앙값으로 대체
   - Q1, Q3, IQR 값 직접 계산하여 처리

4. **데이터 타입 오류 수정**
   - ExamDate 컬럼: 문자열을 datetime으로 변환

## 힌트

- 결측치 확인: `df.isnull().sum()`
- 중복 확인: `df.duplicated(subset=['컬럼명']).sum()`
- 이상치 탐지: IQR 방법 (Q1, Q3, IQR = Q3-Q1, 경계 = Q1±1.5*IQR)
- 타입 변환: `pd.to_numeric(errors='coerce')`, `pd.to_datetime()`

## 출력 형식
```
정답코드 출력 참고
```



## 💻 구현해보세요



In [1]:
import sys
print(sys.executable)

/mnt/c/Users/sdh08/PycharmProjects/PythonProject1/.venv/bin/python


In [2]:
import pandas as pd

# 문제가 있는 예제 데이터 생성
# 기본 환자 데이터

# [환자 고유번호, 나이, 성별, 체질량지수, 수축기혈압, 콜레스테롤, 공복혈당, 검사일자, 심혈관질환 여부]
data = {
    'PatientID': ['P001', 'P002', 'P003', 'P004', 'P005', 'P006', 'P007', 'P008', 'P009', 'P010', 'P001', 'P002'],  # 환자 고유번호 - P001, P002 중복
    'Age': [25, 34, None, 45, 52, 67, 30, 45, 41, 58, 25, 34],  # 나이 - None 결측치
    'Sex': [0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1],  # 성별 (0: 남성, 1: 여성)
    'BMI': [22.5, 'NULL', 28.3, 31.2, 24.8, 29.1, 26.5, 21.3, 26.7, 33.4, 22.5, 'NULL'],  # 체질량지수 - 'NULL' 타입오류
    'BloodPressure': [120, 130, 125, None, 140, 135, 150, 400, None, 160, 120, 130],  # 수축기혈압 - None 결측치, 400 이상치
    'Cholesterol': [180, 220, None, 240, 200, 250, 190, 210, 230, 250, 180, 220],  # 콜레스테롤 - None 결측치
    'BloodSugar': [95, 110, 88, 105, 140, 105, 102, 125, 90, 100, 95, 110],  # 공복혈당
    'ExamDate': ['2023-01-01', '2023-01-02', '2023-01-03', '2023-01-04', '2023-01-05',
                '2023-01-06', '2023-01-07', '2023-01-08', '2023-01-09', '2023-01-10',
                '2023-02-01', '2023-02-02'],  # 검사일자 - 문자열 타입, P001과 P002가 재검사
    'HeartDisease': [0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1]  # 심혈관질환 여부 (0: 없음, 1: 있음)
}


df = pd.DataFrame(data)

print("문제가 있는 의료 데이터:")
print(f"데이터 크기: {df.shape}")

df.head()


문제가 있는 의료 데이터:
데이터 크기: (12, 9)


,PatientID,Age,Sex,BMI,BloodPressure,Cholesterol,BloodSugar,ExamDate,HeartDisease
0,P001,25.0,0,22.5,120.0,180.0,95,2023-01-01,0
1,P002,34.0,1,NULL,130.0,220.0,110,2023-01-02,1
2,P003,NaN,0,28.3,125.0,NaN,88,2023-01-03,0
3,P004,45.0,1,31.2,NaN,240.0,105,2023-01-04,1
4,P005,52.0,1,24.8,140.0,200.0,140,2023-01-05,1


In [2]:
# 원본 데이터 보존을 위해 복사본 생성
import numpy as np
df_processed = df.copy()

# 1. 결측치 탐지 및
print("=== 1. 결측치 탐지 및 처리 ===")

#문자열 결측지들을 np.nan으로 변환
df_processed = df_processed.replace(
    ["null", "NULL", "", "NA", "n/a"],
    np.nan
)
print("결측치 현황:")
null_count = df_processed.isnull().sum()
print(null_count[null_count > 0])
# - Age 결측치: 평균값으로 대체

mean_age = df_processed['Age'].mean()
df_processed['Age']=df_processed['Age'].fillna(value=mean_age)

# - BMI 결측치: 중앙값으로 대체 (타입 변환 후)
median_bmi = df_processed['BMI'].median()
df_processed['BMI']=df_processed['BMI'].fillna(median_bmi)

# - BloodPressure 결측치: 평균값으로 대체
mean_bp = df_processed['BloodPressure'].mean()
df_processed['BloodPressure'] = df_processed['BloodPressure'].fillna(mean_bp)
# - Cholesterol 결측치: 중앙값으로 대체
median_cholesterol = df_processed['Cholesterol'].median()
df_processed['Cholesterol'] = df_processed['Cholesterol'].fillna(median_cholesterol)

print("\n결측치 처리 완료!")

# 2. 중복 데이터 처리
print("\n=== 2. 중복 데이터 처리 ===")
is_duplicate = df_processed.duplicated(subset=['PatientID'])
print(f"중복 데이터: {is_duplicate.sum()}개")
# print(df_processed[~is_duplicate])
print(f"중복 제거 전 데이터 크기: {df_processed.shape}")
df_processed.drop_duplicates(subset=['PatientID'], inplace=True,keep='last')
print(f"중복 제거 후 데이터 크기: {df_processed.shape}")

# 3. 이상치 탐지 및 처리
print("\n=== 3. 이상치 탐지 및 처리 ===")
search_list = ['Age','BMI','BloodPressure','Cholesterol','BloodSugar']

#blood sugar 컬럼이 int만 받아서 median을 넣으려하면 error가 남 -> search_list의 컬럼들 flaot으로 형변환
df_processed[search_list] = df_processed[search_list].astype(float)

for item in search_list:
    Q1 = df_processed[item].quantile(0.25)
    Q3 = df_processed[item].quantile(0.75)
    IQR = Q3 - Q1
    median_value = df_processed[item].median()
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    print(f"\n{item} IQR 범위: {lower_bound} ~ {upper_bound}")
    outliers_condition = (df_processed[item]<lower_bound) | (df_processed[item]>upper_bound)
    print(f"{item} 이상치: {outliers_condition.sum()}개")
    if outliers_condition.any():
        df_processed.loc[outliers_condition, item] = median_value
        print(f"이상치를 중앙값({median_value})으로 대체 완료\n")
#blood sugar 140이 이상치로 판단돼 대체됨, IQR계수를 늘리거나 도메인기준 설정 필요.
# 4. 데이터 타입 오류 수정
print(f"변경 전 ExamDate 데이터타입: {df_processed['ExamDate'].dtype}")
print("ExamDate 데이터타입 변경 [str -> date_time]")
df_processed['ExamDate'] = pd.to_datetime(df_processed['ExamDate'])
print(f"변경 후 ExamDate 데이터타입: {df_processed['ExamDate'].dtype}")

print("\n=== 최종 결과 ===")
print("결측치:", df_processed.isnull().sum().sum())
print("중복 데이터:", df_processed.duplicated(subset=['PatientID']).sum())
print("데이터 크기:", df_processed.shape)

print("\n데이터 전처리 완료 후 최종 데이터:\n")
df_processed


=== 1. 결측치 탐지 및 처리 ===
결측치 현황:
Age              1
BMI              2
BloodPressure    2
Cholesterol      1
dtype: int64

결측치 처리 완료!

=== 2. 중복 데이터 처리 ===
중복 데이터: 2개
중복 제거 전 데이터 크기: (12, 9)
중복 제거 후 데이터 크기: (10, 9)

=== 3. 이상치 탐지 및 처리 ===

Age IQR 범위: 14.0 ~ 72.0
Age 이상치: 0개

BMI IQR 범위: 19.7125 ~ 34.4125
BMI 이상치: 0개

BloodPressure IQR 범위: 87.0 ~ 205.0
BloodPressure 이상치: 1개
이상치를 중앙값(145.0)으로 대체 완료


Cholesterol IQR 범위: 150.0 ~ 290.0
Cholesterol 이상치: 0개

BloodSugar IQR 범위: 77.5 ~ 127.5
BloodSugar 이상치: 1개
이상치를 중앙값(103.5)으로 대체 완료

변경 전 ExamDate 데이터타입: str
ExamDate 데이터타입 변경 [str -> date_time]
변경 후 ExamDate 데이터타입: datetime64[us]

=== 최종 결과 ===
결측치: 0
중복 데이터: 0
데이터 크기: (10, 9)

데이터 전처리 완료 후 최종 데이터:



,PatientID,Age,Sex,BMI,BloodPressure,Cholesterol,BloodSugar,ExamDate,HeartDisease
2,P003,41.454545,0,28.3,125.0,220.0,88.0,2023-01-03,0
3,P004,45.000000,1,31.2,161.0,240.0,105.0,2023-01-04,1
4,P005,52.000000,1,24.8,140.0,200.0,103.5,2023-01-05,1
5,P006,67.000000,0,29.1,135.0,250.0,105.0,2023-01-06,1
6,P007,30.000000,1,26.5,150.0,190.0,102.0,2023-01-07,0
7,P008,45.000000,0,21.3,145.0,210.0,125.0,2023-01-08,0
8,P009,41.000000,1,26.7,161.0,230.0,90.0,2023-01-09,1
9,P010,58.000000,0,33.4,160.0,250.0,100.0,2023-01-10,1
10,P001,25.000000,0,22.5,120.0,180.0,95.0,2023-02-01,0
11,P002,34.000000,1,26.6,130.0,220.0,110.0,2023-02-02,1


In [66]:
# @title 정답

import pandas as pd
import numpy as np

# 원본 데이터 보존을 위해 복사본 생성
df_processed = df.copy()

# ========================================
# 1. 결측치 탐지 및 처리
# ========================================

print("=== 1. 결측치 탐지 및 처리 ===")

# 결측치 현황 확인
print("결측치 현황:")
missing_count = df_processed.isnull().sum()
print(missing_count[missing_count > 0])

# Age: 평균값으로 대체
age_mean = df_processed['Age'].mean()
df_processed['Age'] = df_processed['Age'].fillna(age_mean)

# BMI 타입 변환 후 결측치 처리: 중앙값으로 대체
df_processed['BMI'] = pd.to_numeric(df_processed['BMI'], errors='coerce')
bmi_median = df_processed['BMI'].median()
df_processed['BMI'] = df_processed['BMI'].fillna(bmi_median)

# BloodPressure: 평균값으로 대체
bp_mean = df_processed['BloodPressure'].mean()
df_processed['BloodPressure'] = df_processed['BloodPressure'].fillna(bp_mean)

# Cholesterol: 중앙값으로 대체
chol_median = df_processed['Cholesterol'].median()
df_processed['Cholesterol'] = df_processed['Cholesterol'].fillna(chol_median)

print("결측치 처리 완료!")

# ========================================
# 2. 중복 데이터 처리
# ========================================

print("\n=== 2. 중복 데이터 처리 ===")

# 중복 확인
duplicate_count = df_processed.duplicated(subset=['PatientID']).sum()
print(f"중복 데이터: {duplicate_count}개")

print(f"중복 제거 전 데이터 크기: {df_processed.shape}")

# 중복 제거 (최신 데이터 유지)
df_processed = df_processed.drop_duplicates(subset=['PatientID'], keep='last')

# PatientID 기준으로 정렬
df_processed = df_processed.sort_values('PatientID').reset_index(drop=True)

print(f"중복 제거 후 데이터 크기: {df_processed.shape}")

# ========================================
# 3. 이상치 탐지 및 처리
# ========================================

print("\n=== 3. 이상치 탐지 및 처리 ===")

# BloodPressure 이상치 처리 (IQR 방법)
Q1 = df_processed['BloodPressure'].quantile(0.25)
Q3 = df_processed['BloodPressure'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"BloodPressure IQR 범위: {lower_bound:.1f} ~ {upper_bound:.1f}")

# 이상치 개수 확인
outliers_condition = (df_processed['BloodPressure'] < lower_bound) | (df_processed['BloodPressure'] > upper_bound)
bp_outliers = outliers_condition.sum()
print(f"혈압 이상치: {bp_outliers}개")

# 이상치를 중앙값으로 대체
bp_median = df_processed['BloodPressure'].median()
df_processed.loc[outliers_condition, 'BloodPressure'] = bp_median

print(f"이상치를 중앙값({bp_median:.1f})으로 대체 완료")

# ========================================
# 4. 데이터 타입 오류 수정
# ========================================

print("\n=== 4. 데이터 타입 오류 수정 ===")

# ExamDate 컬럼 (문자열 → datetime)
df_processed['ExamDate'] = pd.to_datetime(df_processed['ExamDate'])

print("데이터 타입 수정 완료!")
print(f"ExamDate 데이터타입: {df_processed['ExamDate'].dtype}")

# 최종 결과 확인
print("\n=== 최종 결과 ===")
print("결측치:", df_processed.isnull().sum().sum())
print("중복 데이터:", df_processed.duplicated(subset=['PatientID']).sum())
print("데이터 크기:", df_processed.shape)

print("\n데이터 전처리 완료 후 최종 데이터:\n")
df_processed

=== 1. 결측치 탐지 및 처리 ===
결측치 현황:
Age              1
BloodPressure    2
Cholesterol      1
dtype: int64
결측치 처리 완료!

=== 2. 중복 데이터 처리 ===
중복 데이터: 2개
중복 제거 전 데이터 크기: (12, 9)
중복 제거 후 데이터 크기: (10, 9)

=== 3. 이상치 탐지 및 처리 ===
BloodPressure IQR 범위: 87.0 ~ 205.0
혈압 이상치: 1개
이상치를 중앙값(145.0)으로 대체 완료

=== 4. 데이터 타입 오류 수정 ===
데이터 타입 수정 완료!
ExamDate 데이터타입: datetime64[us]

=== 최종 결과 ===
결측치: 0
중복 데이터: 0
데이터 크기: (10, 9)

데이터 전처리 완료 후 최종 데이터:



,PatientID,Age,Sex,BMI,BloodPressure,Cholesterol,BloodSugar,ExamDate,HeartDisease
0,P001,25.000000,0,22.5,120.0,180.0,95,2023-02-01,0
1,P002,34.000000,1,26.6,130.0,220.0,110,2023-02-02,1
2,P003,41.454545,0,28.3,125.0,220.0,88,2023-01-03,0
3,P004,45.000000,1,31.2,161.0,240.0,105,2023-01-04,1
4,P005,52.000000,1,24.8,140.0,200.0,140,2023-01-05,1
5,P006,67.000000,0,29.1,135.0,250.0,105,2023-01-06,1
6,P007,30.000000,1,26.5,150.0,190.0,102,2023-01-07,0
7,P008,45.000000,0,21.3,145.0,210.0,125,2023-01-08,0
8,P009,41.000000,1,26.7,161.0,230.0,90,2023-01-09,1
9,P010,58.000000,0,33.4,160.0,250.0,100,2023-01-10,1


In [11]:
# @title 정답

import pandas as pd
import numpy as np

# 원본 데이터 보존을 위해 복사본 생성
df_processed = df.copy()

# ========================================
# 1. 결측치 탐지 및 처리
# ========================================

print("=== 1. 결측치 탐지 및 처리 ===")

# 결측치 현황 확인
print("결측치 현황:")
missing_count = df_processed.isnull().sum()
print(missing_count[missing_count > 0])

# Age: 평균값으로 대체
age_mean = df_processed['Age'].mean()
df_processed['Age'] = df_processed['Age'].fillna(age_mean)

# BMI 타입 변환 후 결측치 처리: 중앙값으로 대체
df_processed['BMI'] = pd.to_numeric(df_processed['BMI'], errors='coerce')
bmi_median = df_processed['BMI'].median()
df_processed['BMI'] = df_processed['BMI'].fillna(bmi_median)

# BloodPressure: 평균값으로 대체
bp_mean = df_processed['BloodPressure'].mean()
df_processed['BloodPressure'] = df_processed['BloodPressure'].fillna(bp_mean)

# Cholesterol: 중앙값으로 대체
chol_median = df_processed['Cholesterol'].median()
df_processed['Cholesterol'] = df_processed['Cholesterol'].fillna(chol_median)

print("결측치 처리 완료!")

# ========================================
# 2. 중복 데이터 처리
# ========================================

print("\n=== 2. 중복 데이터 처리 ===")

# 중복 확인
duplicate_count = df_processed.duplicated(subset=['PatientID']).sum()
print(f"중복 데이터: {duplicate_count}개")

print(f"중복 제거 전 데이터 크기: {df_processed.shape}")

# 중복 제거 (최신 데이터 유지)
df_processed = df_processed.drop_duplicates(subset=['PatientID'], keep='last')

# PatientID 기준으로 정렬
df_processed = df_processed.sort_values('PatientID').reset_index(drop=True)

print(f"중복 제거 후 데이터 크기: {df_processed.shape}")

# ========================================
# 3. 이상치 탐지 및 처리
# ========================================

print("\n=== 3. 이상치 탐지 및 처리 ===")

# BloodPressure 이상치 처리 (IQR 방법)
Q1 = df_processed['BloodPressure'].quantile(0.25)
Q3 = df_processed['BloodPressure'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"BloodPressure IQR 범위: {lower_bound:.1f} ~ {upper_bound:.1f}")

# 이상치 개수 확인
outliers_condition = (df_processed['BloodPressure'] < lower_bound) | (df_processed['BloodPressure'] > upper_bound)
bp_outliers = outliers_condition.sum()
print(f"혈압 이상치: {bp_outliers}개")

# 이상치를 중앙값으로 대체
bp_median = df_processed['BloodPressure'].median()
df_processed.loc[outliers_condition, 'BloodPressure'] = bp_median

print(f"이상치를 중앙값({bp_median:.1f})으로 대체 완료")

# ========================================
# 4. 데이터 타입 오류 수정
# ========================================

print("\n=== 4. 데이터 타입 오류 수정 ===")

# ExamDate 컬럼 (문자열 → datetime)
df_processed['ExamDate'] = pd.to_datetime(df_processed['ExamDate'])

print("데이터 타입 수정 완료!")
print(f"ExamDate 데이터타입: {df_processed['ExamDate'].dtype}")

# 최종 결과 확인
print("\n=== 최종 결과 ===")
print("결측치:", df_processed.isnull().sum().sum())
print("중복 데이터:", df_processed.duplicated(subset=['PatientID']).sum())
print("데이터 크기:", df_processed.shape)

print("\n데이터 전처리 완료 후 최종 데이터:\n")
df_processed

=== 1. 결측치 탐지 및 처리 ===
결측치 현황:
Age              1
BloodPressure    2
Cholesterol      1
dtype: int64
결측치 처리 완료!

=== 2. 중복 데이터 처리 ===
중복 데이터: 2개
중복 제거 전 데이터 크기: (12, 9)
중복 제거 후 데이터 크기: (10, 9)

=== 3. 이상치 탐지 및 처리 ===
BloodPressure IQR 범위: 87.0 ~ 205.0
혈압 이상치: 1개
이상치를 중앙값(145.0)으로 대체 완료

=== 4. 데이터 타입 오류 수정 ===
데이터 타입 수정 완료!
ExamDate 데이터타입: datetime64[us]

=== 최종 결과 ===
결측치: 0
중복 데이터: 0
데이터 크기: (10, 9)

데이터 전처리 완료 후 최종 데이터:



,PatientID,Age,Sex,BMI,BloodPressure,Cholesterol,BloodSugar,ExamDate,HeartDisease
0,P001,25.000000,0,22.5,120.0,180.0,95,2023-02-01,0
1,P002,34.000000,1,26.6,130.0,220.0,110,2023-02-02,1
2,P003,41.454545,0,28.3,125.0,220.0,88,2023-01-03,0
3,P004,45.000000,1,31.2,161.0,240.0,105,2023-01-04,1
4,P005,52.000000,1,24.8,140.0,200.0,140,2023-01-05,1
5,P006,67.000000,0,29.1,135.0,250.0,105,2023-01-06,1
6,P007,30.000000,1,26.5,150.0,190.0,102,2023-01-07,0
7,P008,45.000000,0,21.3,145.0,210.0,125,2023-01-08,0
8,P009,41.000000,1,26.7,161.0,230.0,90,2023-01-09,1
9,P010,58.000000,0,33.4,160.0,250.0,100,2023-01-10,1
